# Init

In [ ]:
!python -m spacy download en_core_web_sm

In [17]:
import os
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import spacy

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean

DOCS_FOLDER_PATH = os.path.join(os.path.abspath('.'), 'docs')

# Parce and cleaned up docs

In [24]:
class TextParser:
    def __init__(self, category_mapping_path: str, folder_path: str = DOCS_FOLDER_PATH, files_format: str = 'txt'):
        self.category_mapping_path = category_mapping_path
        self.folder_path = folder_path
        self.files_format = files_format

    def parce(self) -> pd.DataFrame:
        docs = self.get_files_data()
        categories = pd.read_csv(self.category_mapping_path)
        return docs.merge(categories, on="file_name")

    def get_files_data(self) -> pd.DataFrame:
        docs = {}
        for file_name in os.listdir(self.folder_path):
            if not file_name.endswith(self.files_format):
                continue
            with open(os.path.join(self.folder_path, file_name)) as f:
                docs[file_name] = f.read()
        return pd.DataFrame(docs.items(), columns=['file_name', 'raw_text'])


t_p = TextParser(category_mapping_path=os.path.join(DOCS_FOLDER_PATH, 'document_category_mapping.csv'))
docs_df = t_p.parce()

,file_name,raw_text,category
0,doc_4.txt,match team team coach team coach stadium playe...,sports
1,doc_7.txt,network computer storage network network algor...,tech
2,doc_3.txt,league player player team team training league...,sports
3,doc_8.txt,match tournament league training football matc...,sports
4,doc_10.txt,code python performance storage storage algori...,tech
5,doc_1.txt,election policy policy law minister reform ele...,politics
6,doc_2.txt,goal training player football football stadium...,sports
7,doc_9.txt,performance data algorithm computer performanc...,tech
8,doc_5.txt,match training stadium team coach goal trainin...,sports
9,doc_6.txt,football stadium training football match footb...,sports


In [5]:
class TextPreprocessor:
    def __init__(self, lang_model: str = "en_core_web_sm"):
        self.nlp = spacy.load(lang_model)

    def preprocess_text(self, text: str) -> list[str]:
        text = self._remove_non_letters(text)
        return self._lemmatize(text)

    def preprocess_documents(self, docs: dict[str, str]) -> dict[str, list[str]]:
        return {name: self.preprocess_text(text) for name, text in docs.items()}

    @staticmethod
    def _remove_non_letters(text: str) -> str:
        text = text.lower()
        return re.sub(r'[^a-z\s]', '', text)

    def _lemmatize(self, text: str) -> list[str]:
        return [token.lemma_ for token in self.nlp(text) if token.is_alpha and not token.is_stop]


preprocessor = TextPreprocessor()
docs_df["tokens"] = docs_df["raw_text"].apply(preprocessor.preprocess_text)
docs_df

,file_name,raw_text,category,tokens
0,doc_4.txt,match team team coach team coach stadium playe...,sports,"[match, team, team, coach, team, coach, stadiu..."
1,doc_7.txt,network computer storage network network algor...,tech,"[network, computer, storage, network, network,..."
2,doc_3.txt,league player player team team training league...,sports,"[league, player, player, team, team, training,..."
3,doc_8.txt,match tournament league training football matc...,sports,"[match, tournament, league, training, football..."
4,doc_10.txt,code python performance storage storage algori...,tech,"[code, python, performance, storage, storage, ..."
5,doc_1.txt,election policy policy law minister reform ele...,politics,"[election, policy, policy, law, minister, refo..."
6,doc_2.txt,goal training player football football stadium...,sports,"[goal, training, player, football, football, s..."
7,doc_9.txt,performance data algorithm computer performanc...,tech,"[performance, datum, algorithm, computer, perf..."
8,doc_5.txt,match training stadium team coach goal trainin...,sports,"[match, training, stadium, team, coach, goal, ..."
9,doc_6.txt,football stadium training football match footb...,sports,"[football, stadium, training, football, match,..."


# Metrics analyses

In [6]:
class FrequencyAnalysis:
    def __init__(self, df: pd.DataFrame):
        """
        :param df: DataFrame with columns ['file_name', 'tokens', 'category']
        """

        self.df = df.copy()
        self.vectorizer = CountVectorizer()
        self.word_doc_matrix = None
        self.doc_names = df["file_name"].tolist()
        self.categories = df["category"].tolist()
        self.term_names = []

    def build_word_document_matrix(self) -> pd.DataFrame:
        docs = self.df["tokens"].apply(lambda tokens: " ".join(tokens))
        X = self.vectorizer.fit_transform(docs)
        self.word_doc_matrix = pd.DataFrame(
            X.toarray(),
            index=self.doc_names,
            columns=self.vectorizer.get_feature_names_out()
        )
        self.term_names = self.vectorizer.get_feature_names_out().tolist()
        return self.word_doc_matrix

    def build_document_category_matrix(self) -> pd.DataFrame:
        unique_categories = sorted(set(self.categories))
        matrix = np.zeros((len(self.df), len(unique_categories)))
        category_index = {cat: i for i, cat in enumerate(unique_categories)}
        for i, cat in enumerate(self.categories):
            matrix[i][category_index[cat]] = 1
        return pd.DataFrame(matrix, index=self.doc_names, columns=unique_categories)

    def build_word_word_matrix(self) -> pd.DataFrame:
        if self.word_doc_matrix is None:
            self.build_word_document_matrix()
        matrix = np.dot(self.word_doc_matrix.T, self.word_doc_matrix)
        return pd.DataFrame(matrix, index=self.term_names, columns=self.term_names)

    @staticmethod
    def compute_euclidean_distances(matrix: pd.DataFrame) -> pd.DataFrame:
        dist_matrix = pd.DataFrame(index=matrix.index, columns=matrix.index, dtype=float)
        for i in matrix.index:
            for j in matrix.index:
                dist_matrix.loc[i, j] = euclidean(matrix.loc[i], matrix.loc[j])
        return dist_matrix

    @staticmethod
    def compute_cosine_similarities(matrix: pd.DataFrame) -> pd.DataFrame:
        cos_sim = cosine_similarity(matrix.values)
        return pd.DataFrame(cos_sim, index=matrix.index, columns=matrix.index)


fa = FrequencyAnalysis(docs_df)

word_doc = fa.build_word_document_matrix()
word_word = fa.build_word_word_matrix()

doc_euclidean = fa.compute_euclidean_distances(word_doc)
doc_cosine = fa.compute_cosine_similarities(word_doc)

word_euclidean = fa.compute_euclidean_distances(word_word)
word_cosine = fa.compute_cosine_similarities(word_word)

In [25]:
word_doc

,ai,algorithm,coach,code,computer,datum,debate,election,football,goal,...,president,python,reform,stadium,storage,system,team,tournament,train,training
doc_4.txt,0,0,5,0,0,0,0,0,4,2,...,0,0,0,2,0,0,5,2,1,2
doc_7.txt,2,3,0,2,1,2,0,0,0,0,...,0,2,0,0,2,1,0,0,0,0
doc_3.txt,0,0,1,0,0,0,0,0,1,2,...,0,0,0,2,0,0,5,2,1,2
doc_8.txt,0,0,1,0,0,0,0,0,2,2,...,0,0,0,4,0,0,3,1,0,5
doc_10.txt,4,5,0,1,3,2,0,0,0,0,...,0,2,0,0,6,0,0,0,0,0
doc_1.txt,0,0,0,0,0,0,1,5,0,0,...,3,0,3,0,0,0,0,0,0,0
doc_2.txt,0,0,3,0,0,0,0,0,4,2,...,0,0,0,4,0,0,3,2,0,2
doc_9.txt,2,6,0,1,2,2,0,0,0,0,...,0,2,0,0,2,4,0,0,0,0
doc_5.txt,0,0,3,0,0,0,0,0,2,2,...,0,0,0,4,0,0,2,2,0,3
doc_6.txt,0,0,3,0,0,0,0,0,3,1,...,0,0,0,2,0,0,3,3,0,2


In [29]:
word_euclidean

,ai,algorithm,coach,code,computer,datum,debate,election,football,goal,...,president,python,reform,stadium,storage,system,team,tournament,train,training
ai,0.000000,53.469618,151.525575,40.718546,18.627936,21.908902,75.226325,85.807925,150.824401,116.614750,...,78.911343,21.908902,78.911343,161.950610,21.908902,37.509999,189.678676,121.568910,77.142725,149.786515
algorithm,53.469618,0.000000,183.103796,92.978492,70.964780,73.776690,127.381317,133.902950,182.523971,155.454173,...,129.591666,73.776690,129.591666,191.820228,35.142567,85.486841,215.742903,159.204271,128.522372,181.667278
coach,151.525575,183.103796,0.000000,136.301137,143.384100,142.267354,132.071950,138.372685,6.324555,44.777226,...,134.205067,142.267354,134.205067,24.413111,163.217646,138.920841,48.041649,37.215588,114.302231,27.964263
code,40.718546,92.978492,136.301137,0.000000,23.086793,19.235384,35.735137,54.598535,135.521216,96.005208,...,42.953463,19.235384,42.953463,147.803924,62.498000,16.401219,177.752637,101.965681,39.610605,134.365174
computer,18.627936,70.964780,143.384100,23.086793,0.000000,5.916080,57.078893,70.441465,142.642911,105.820603,...,61.854668,5.916080,61.854668,154.359969,40.236799,20.976177,183.240279,111.256460,59.581876,141.545046
datum,21.908902,73.776690,142.267354,19.235384,5.916080,0.000000,54.212545,68.139563,141.520317,104.302445,...,59.219929,0.000000,59.219929,153.323188,43.817805,17.635192,182.367760,109.813478,56.841886,140.413675
debate,75.226325,127.381317,132.071950,35.735137,57.078893,54.212545,0.000000,33.704599,131.266904,89.899944,...,16.852300,54.212545,16.852300,143.913168,96.638502,44.698993,174.530800,96.239285,20.832667,130.073056
election,85.807925,133.902950,138.372685,54.598535,70.441465,68.139563,33.704599,0.000000,137.604506,98.924213,...,16.852300,68.139563,16.852300,149.716399,105.085679,60.844063,179.346034,104.718671,46.238512,136.466113
football,150.824401,182.523971,6.324555,135.521216,142.642911,141.520317,131.266904,137.604506,0.000000,43.370497,...,133.412893,141.520317,133.412893,20.396078,162.566909,138.155709,48.641546,36.235342,113.688170,22.538855
goal,116.614750,155.454173,44.777226,96.005208,105.820603,104.302445,89.899944,98.924213,43.370497,0.000000,...,93.005376,104.302445,93.005376,55.090834,131.449610,99.689518,85.340494,9.055385,71.833140,43.092923


# Docs analyses

In [18]:
class TermImportanceCalculator:
    def __init__(self, word_doc_matrix: pd.DataFrame, df: pd.DataFrame):
        """
        :param word_doc_matrix: DataFrame, rows=documents, columns=terms (частоти термів)
        :param df: Original DataFrame with 'file_name' and 'category' columns
        """
        self.word_doc = word_doc_matrix
        self.df = df.set_index("file_name").loc[word_doc_matrix.index]
        self.tf = word_doc_matrix.copy()
        self.idf = None
        self.tfidf = None
        self.tf_slf = None

    def compute_tf(self) -> pd.DataFrame:
        tf_normalized = self.tf.div(self.tf.sum(axis=1), axis=0)
        return tf_normalized

    def compute_idf(self) -> pd.Series:
        N = len(self.tf)
        df_counts = (self.tf > 0).sum(axis=0)
        idf = np.log((N + 1) / (df_counts + 1)) + 1
        self.idf = idf
        return idf

    def compute_tfidf(self) -> pd.DataFrame:
        tf = self.compute_tf()
        idf = self.compute_idf()
        self.tfidf = tf * idf
        return self.tfidf

    def compute_tf_slf(self) -> pd.DataFrame:
        term_category_counts = defaultdict(lambda: defaultdict(int))
        category_counts = defaultdict(int)

        for doc_id, row in self.tf.iterrows():
            category = self.df.loc[doc_id, "category"]
            category_counts[category] += 1
            for term in row[row > 0].index:
                term_category_counts[term][category] += 1

        tf_slf_values = {}
        for term in self.tf.columns:
            term_cat_map = term_category_counts[term]
            if not term_cat_map:
                tf_slf_values[term] = 0
                continue
            category_ratios = [
                term_cat_map[cat] / category_counts[cat]
                for cat in term_cat_map
            ]
            max_ratio = max(category_ratios)
            spread_penalty = len(term_cat_map)
            tf_slf_values[term] = max_ratio / spread_penalty

        self.tf_slf = pd.Series(tf_slf_values)
        return self.tf_slf.sort_values(ascending=False)


calc = TermImportanceCalculator(word_doc_matrix=word_doc, df=docs_df)

tf = calc.compute_tf()
idf = calc.compute_idf()
tfidf = calc.compute_tfidf()
tf_slf = calc.compute_tf_slf()

In [26]:
tf

,ai,algorithm,coach,code,computer,datum,debate,election,football,goal,...,president,python,reform,stadium,storage,system,team,tournament,train,training
doc_4.txt,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,0.133333,0.066667,...,0.000000,0.000000,0.000000,0.066667,0.000000,0.000000,0.166667,0.066667,0.033333,0.066667
doc_7.txt,0.086957,0.130435,0.000000,0.086957,0.043478,0.086957,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.086957,0.000000,0.000000,0.086957,0.043478,0.000000,0.000000,0.000000,0.000000
doc_3.txt,0.000000,0.000000,0.037037,0.000000,0.000000,0.000000,0.000000,0.000000,0.037037,0.074074,...,0.000000,0.000000,0.000000,0.074074,0.000000,0.000000,0.185185,0.074074,0.037037,0.074074
doc_8.txt,0.000000,0.000000,0.038462,0.000000,0.000000,0.000000,0.000000,0.000000,0.076923,0.076923,...,0.000000,0.000000,0.000000,0.153846,0.000000,0.000000,0.115385,0.038462,0.000000,0.192308
doc_10.txt,0.133333,0.166667,0.000000,0.033333,0.100000,0.066667,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.066667,0.000000,0.000000,0.200000,0.000000,0.000000,0.000000,0.000000,0.000000
doc_1.txt,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.047619,0.238095,0.000000,0.000000,...,0.142857,0.000000,0.142857,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
doc_2.txt,0.000000,0.000000,0.125000,0.000000,0.000000,0.000000,0.000000,0.000000,0.166667,0.083333,...,0.000000,0.000000,0.000000,0.166667,0.000000,0.000000,0.125000,0.083333,0.000000,0.083333
doc_9.txt,0.076923,0.230769,0.000000,0.038462,0.076923,0.076923,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.076923,0.000000,0.000000,0.076923,0.153846,0.000000,0.000000,0.000000,0.000000
doc_5.txt,0.000000,0.000000,0.136364,0.000000,0.000000,0.000000,0.000000,0.000000,0.090909,0.090909,...,0.000000,0.000000,0.000000,0.181818,0.000000,0.000000,0.090909,0.090909,0.000000,0.136364
doc_6.txt,0.000000,0.000000,0.130435,0.000000,0.000000,0.000000,0.000000,0.000000,0.130435,0.043478,...,0.000000,0.000000,0.000000,0.086957,0.000000,0.000000,0.130435,0.130435,0.000000,0.086957


In [27]:
idf

ai             2.011601
algorithm      2.011601
coach          1.451985
code           2.011601
computer       2.011601
datum          2.011601
debate         2.704748
election       2.704748
football       1.451985
goal           1.451985
government     2.704748
law            2.704748
league         1.451985
match          1.788457
minister       2.704748
network        2.011601
performance    2.011601
player         1.451985
policy         2.704748
president      2.704748
python         2.011601
reform         2.704748
stadium        1.451985
storage        2.011601
system         2.299283
team           1.451985
tournament     1.451985
train          2.299283
training       1.451985
dtype: float64

In [28]:
tfidf

,ai,algorithm,coach,code,computer,datum,debate,election,football,goal,...,president,python,reform,stadium,storage,system,team,tournament,train,training
doc_4.txt,0.000000,0.000000,0.241998,0.000000,0.000000,0.000000,0.000000,0.000000,0.193598,0.096799,...,0.000000,0.000000,0.000000,0.096799,0.000000,0.000000,0.241998,0.096799,0.076643,0.096799
doc_7.txt,0.174922,0.262383,0.000000,0.174922,0.087461,0.174922,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.174922,0.000000,0.000000,0.174922,0.099969,0.000000,0.000000,0.000000,0.000000
doc_3.txt,0.000000,0.000000,0.053777,0.000000,0.000000,0.000000,0.000000,0.000000,0.053777,0.107554,...,0.000000,0.000000,0.000000,0.107554,0.000000,0.000000,0.268886,0.107554,0.085159,0.107554
doc_8.txt,0.000000,0.000000,0.055846,0.000000,0.000000,0.000000,0.000000,0.000000,0.111691,0.111691,...,0.000000,0.000000,0.000000,0.223382,0.000000,0.000000,0.167537,0.055846,0.000000,0.279228
doc_10.txt,0.268213,0.335267,0.000000,0.067053,0.201160,0.134107,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.134107,0.000000,0.000000,0.402320,0.000000,0.000000,0.000000,0.000000,0.000000
doc_1.txt,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.128798,0.643988,0.000000,0.000000,...,0.386393,0.000000,0.386393,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
doc_2.txt,0.000000,0.000000,0.181498,0.000000,0.000000,0.000000,0.000000,0.000000,0.241998,0.120999,...,0.000000,0.000000,0.000000,0.241998,0.000000,0.000000,0.181498,0.120999,0.000000,0.120999
doc_9.txt,0.154739,0.464216,0.000000,0.077369,0.154739,0.154739,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.154739,0.000000,0.000000,0.154739,0.353736,0.000000,0.000000,0.000000,0.000000
doc_5.txt,0.000000,0.000000,0.197998,0.000000,0.000000,0.000000,0.000000,0.000000,0.131999,0.131999,...,0.000000,0.000000,0.000000,0.263997,0.000000,0.000000,0.131999,0.131999,0.000000,0.197998
doc_6.txt,0.000000,0.000000,0.189389,0.000000,0.000000,0.000000,0.000000,0.000000,0.189389,0.063130,...,0.000000,0.000000,0.000000,0.126260,0.000000,0.000000,0.189389,0.189389,0.000000,0.126260
